# Lab 03 · Prompt chaining with quality gates

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Chain a small DDD design exercise
**Start with a problem:** One large request for service boundaries and a final decision can hide how the answer was reached.

**In this lab:** Break the task into smaller prompts and inspect the handoffs before using their results.

**So far:** Lab 02 retrieved architecture notes.

**New:** prompt chaining with quality gates. Use ShopEasy's events to find business capabilities, propose bounded contexts, compare a business split with a technical-layer split, then write a three-line decision record.

DDD helps identify candidate boundaries; it does not require deploying every candidate as a separate service immediately. A useful boundary keeps related rules together, reduces coordination between services, and gives each context a consistent model.

**At a glance**

![Lab 03 at a glance: Brief, then Business capabilities (3 labels), then gate, then Boundary options (2 options · 3 criteria), then Trade-offs (+ / − table), then gate, then Decision record (Context · Decision · Consequence), then gate.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/03.svg)

## Think about the example

A technical-layer split places UI, business logic, and data access in different services. A bounded context defines where one business model and language apply; its deployment boundary still needs review.

**Concepts:** Event storming; Bounded contexts and ubiquitous language; Business capability vs technical-layer decomposition.

Use the scenario above to consider these questions before running the code:

1. Why can a technical-layer split create a distributed monolith?
2. Why is a bounded context a service candidate rather than an automatic deployment decision?

## The idea in context

A fixed prompt chain exposes intermediate results. Gates can catch a malformed handoff while leaving domain judgement for review.

A bounded context is a model boundary; validate it before choosing deployment boundaries.

| Key term | Meaning |
| --- | --- |
| **Workflow** | A sequence of steps fixed by code; the model does not choose what happens next. |
| **Decomposition** | Splitting a large task into smaller steps with clear inputs and outputs. |
| **Gate** | A plain-code check between two steps that stops a bad result from spreading. |
| **Decision record (ADR)** | A short note of an architecture decision: its context, the decision, and its consequence. |

## Pattern · Single-Path Planning

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Passive Goal Creator.*

| | |
| --- | --- |
| **Context** | Complex goals need several dependent steps, where each result informs the next. |
| **Problem** | One monolithic response cannot reliably manage dependencies and accumulated intermediate results. |
| **Solution** | Make one ordered plan of a few steps, execute each step with the accumulated context, then synthesize the outputs. |
| **Use when** | The work is naturally sequential; steps depend on earlier outputs; one reasonable path is enough. |
| **Trade-off** | Easy to follow, but an early planning mistake propagates through the whole run. |
| **Flow** | Plan → Step 1 → Step N → Synthesize |
| **Quality attributes** | Benefits: Predictability, Performance, Observability, Maintainability. Liabilities: Reliability, Latency, Usability, Adaptability. |
| **In this lab** | Events lead to candidate bounded contexts, then a comparison of boundary strategies and a short decision record. Gates check output structure; domain experts still validate the proposed boundaries. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/5afd3e9402e35da6cba234dd9e794a387595c5c6.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · A brief and a reusable gate
The gates check structural contracts. Students still judge whether a proposed context has coherent language, rules and data. A gate is just a function that returns `None` for a pass or an error message for a failure. `run_step` gets a reply using `ask`, runs the gate, and returns only a checked reply. Later cells reuse those earlier results.

In [ ]:
import json

brief = ("ShopEasy sells jackets. A customer places an order; stock is reserved; payment is authorized; "
         "a parcel is shipped. Order, Inventory and Shipping currently share tables and releases.")
events = ["OrderPlaced", "StockReserved", "PaymentAuthorized", "ParcelShipped"]

def run_step(name, prompt, gate):
    reply = ask(prompt)
    print(name, reply)
    problem = gate(reply)
    if problem:
        raise ValueError(problem)
    return reply

def three_items(reply):
    items = json.loads(reply)
    if not isinstance(items, list) or len(items) != 3:
        return "Expected a JSON list of three items."
    return None

print(events)

## Step 2 · From events to candidate bounded contexts
Event storming names facts in the past tense. Group by business meaning rather than by UI, logic and data-access layers. Our compact exercise treats Payment as an externally provided capability; it does not claim all domains have exactly three contexts.

In [ ]:
prompt = (f"Brief: {brief}\nEvents: {events}\nPropose exactly 3 business bounded contexts for "
          "OrderPlaced, StockReserved and ParcelShipped. Payment is supplied externally. "
          "Reply with a JSON list of context labels only, no code fences or explanation.")
contexts = json.loads(run_step("contexts", prompt, three_items))
options = ["business capability split", "technical layer split"]
criteria = ["cohesive business rules", "independent change", "private data ownership"]
print("Compare:", options, "against", criteria)

## Step 3 · Compare the boundary choices
A technical-layer split can create a distributed monolith: one feature change spans several services. A business split needs stable contracts and private data to earn its independence. The nested loops visit each option–criterion pair. The model’s marks are judgements, not measurements.

In [ ]:
def every_cell(reply):
    table = json.loads(reply)
    if not isinstance(table, dict):
        return "Expected an object."
    for option in options:
        marks = table.get(option, {})
        if not isinstance(marks, dict):
            return f"Expected criterion marks for {option}."
        for criterion in criteria:
            if marks.get(criterion) not in ("+", "-"):
                return f"Missing or invalid mark: {option} / {criterion}."
    return None

prompt = (f"Brief: {brief}\nContexts: {contexts}\nCompare {options} using {criteria}. "
          'JSON only: {"<option>": {"<criterion>": "+" or "-"}}. Marks are qualitative, not measured.')
tradeoffs = json.loads(run_step("trade-offs", prompt, every_cell))

## Step 4 · Record a decision and its cost
The ADR chooses a boundary strategy, not the number of deployed services. Shared identity can cross contracts without sharing the entire internal model. `split(": ", 1)` separates each label from its text once, so a later colon inside the explanation does not create extra fields. The checker enforces labels and the decision option; a human still checks the stated cost.

In [ ]:
def record_gate(reply):
    fields = {}
    for line in reply.strip().splitlines():
        if ": " in line:
            name, value = line.split(": ", 1)
            fields[name] = value
    if list(fields) != ["Context", "Decision", "Consequence"]:
        return "Expected Context, Decision, Consequence."
    if fields["Decision"].strip().lower() not in options:
        return "Unknown boundary strategy."
    return None

prompt = (f"Brief: {brief}\nContexts: {contexts}\nTrade-offs: {json.dumps(tradeoffs)}\n"
          "Write exactly Context:, Decision:, Consequence: lines. Decision is one option label only. "
          "Consequence must name a cost; do not assume the service candidates are already deployed.")
record = run_step("ADR", prompt, record_gate)

## What you should see
The chain proposes contexts, compares both strategies, and records a decision. Three calls to `ask` make the intermediate model responses visible; outputs can vary. Gates verify shape, not the truth of every architectural claim.

## Try it
1. Replace Order, Inventory, Shipping with UI, BusinessLogic, DataAccess. The list gate still passes: explain why the design may still be poor.
2. Explain why `Product` has different fields in Catalog, Inventory and Shipping when Catalog uses title and price, Inventory tracks quantity, and Shipping tracks package weight.
3. A customer returns a damaged jacket; after inspection it is refunded and then restocked or scrapped. Which language, rules and data would you investigate before drawing another boundary?

## Takeaway
A chain makes intermediate modelling choices reviewable. Good JSON is necessary for automation; domain experts and change patterns still validate service candidates.